# Self-Attention Mechanism: Scaled Dot-Product Attention

## Problem Statement
Implement the **scaled dot-product attention** mechanism from scratch, which is the core building block of Transformer architectures. Your implementation should compute attention scores and weighted outputs for input sequences.

## Objectives
1. Implement scaled dot-product attention without using pre-built attention layers
2. Generate random sequences as input data
3. Verify numerical correctness of the implementation
4. Visualize attention scores to understand the attention patterns

## Background
Scaled dot-product attention computes:
```
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
```
where:
- Q (Query), K (Key), V (Value) are input matrices
- d_k is the dimension of the key vectors
- The scaling factor (sqrt(d_k)) prevents gradients from vanishing for large dimensions

## Step 1: Data Acquisition

**Task**: Generate synthetic sequential data for testing the attention mechanism.

**Requirements**:
- Create random Query (Q), Key (K), and Value (V) matrices
- Shape: (batch_size, sequence_length, embedding_dim)
- Use numpy for random generation with a fixed seed for reproducibility
- Typical dimensions: batch_size=2, sequence_length=5, embedding_dim=64

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Define dimensions
batch_size = 2
seq_length = 5
d_model = 64  # embedding dimension

# Generate random Q, K, V matrices
Q = np.random.randn(batch_size, seq_length, d_model)
K = np.random.randn(batch_size, seq_length, d_model)
V = np.random.randn(batch_size, seq_length, d_model)

print(f"Generated input matrices:")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

## Step 2: Data Preprocessing

**Task**: Prepare and validate the input data.

**Requirements**:
- Verify shapes of Q, K, V matrices
- Ensure Q and K have the same embedding dimension (d_k)
- Print shapes and basic statistics of the generated data

In [ ]:
# Validate input data
print("Input Data Validation:")
print("=" * 50)
print(f"Q shape: {Q.shape} - (batch_size, seq_length, d_model)")
print(f"K shape: {K.shape} - (batch_size, seq_length, d_model)")
print(f"V shape: {V.shape} - (batch_size, seq_length, d_model)")
print()

# Verify dimensional compatibility
assert Q.shape[-1] == K.shape[-1], "Q and K must have same embedding dimension"
assert Q.shape[0] == K.shape[0] == V.shape[0], "Batch sizes must match"
assert Q.shape[1] == K.shape[1] == V.shape[1], "Sequence lengths must match"
print("✓ Dimensional compatibility verified")
print()

# Display basic statistics
print("Basic Statistics:")
print(f"Q - Mean: {Q.mean():.4f}, Std: {Q.std():.4f}")
print(f"K - Mean: {K.mean():.4f}, Std: {K.std():.4f}")
print(f"V - Mean: {V.mean():.4f}, Std: {V.std():.4f}")

## Step 3: Model Implementation

**Task**: Implement scaled dot-product attention from scratch.

**Requirements**:
- Implement a function `scaled_dot_product_attention(Q, K, V, mask=None)`
- Compute attention scores: scores = Q @ K^T
- Apply scaling: scores = scores / sqrt(d_k)
- Apply softmax to get attention weights
- Compute weighted sum: output = attention_weights @ V
- Return both output and attention weights
- Support optional masking for padding or causal attention

In [ ]:
def softmax(x, axis=-1):
    """
    Compute softmax along specified axis.
    
    Args:
        x: Input array
        axis: Axis along which to compute softmax
    
    Returns:
        Softmax probabilities
    """
    # Subtract max for numerical stability
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Query matrix of shape (batch_size, seq_length, d_k)
        K: Key matrix of shape (batch_size, seq_length, d_k)
        V: Value matrix of shape (batch_size, seq_length, d_v)
        mask: Optional mask of shape (seq_length, seq_length) or broadcastable
    
    Returns:
        output: Attention output of shape (batch_size, seq_length, d_v)
        attention_weights: Attention weights of shape (batch_size, seq_length, seq_length)
    """
    # Get the dimension of keys for scaling
    d_k = K.shape[-1]
    
    # Compute attention scores: Q @ K^T
    # Shape: (batch_size, seq_length, seq_length)
    scores = np.matmul(Q, K.transpose(0, 2, 1))
    
    # Apply scaling factor
    scores = scores / np.sqrt(d_k)
    
    # Apply mask if provided (set masked positions to large negative value)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Apply softmax to get attention weights
    attention_weights = softmax(scores, axis=-1)
    
    # Compute weighted sum of values
    # Shape: (batch_size, seq_length, d_v)
    output = np.matmul(attention_weights, V)
    
    return output, attention_weights


print("Scaled dot-product attention function implemented successfully!")

## Step 4: Execution and Testing

**Task**: Apply the attention mechanism and verify correctness.

**Requirements**:
- Call the attention function with generated Q, K, V
- Verify output shape matches expected dimensions
- Check that attention weights sum to 1 along the sequence dimension
- Test with a simple example to verify correctness

In [ ]:
# Apply attention mechanism
output, attention_weights = scaled_dot_product_attention(Q, K, V)

print("Attention Mechanism Output:")
print("=" * 50)
print(f"Output shape: {output.shape}")
print(f"Expected shape: (batch_size={batch_size}, seq_length={seq_length}, d_model={d_model})")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"Expected shape: (batch_size={batch_size}, seq_length={seq_length}, seq_length={seq_length})")
print()

# Verify attention weights sum to 1
weights_sum = attention_weights.sum(axis=-1)
print(f"Attention weights sum (should be ~1.0):")
print(f"Min: {weights_sum.min():.6f}")
print(f"Max: {weights_sum.max():.6f}")
print(f"Mean: {weights_sum.mean():.6f}")
print()

# Check if all sums are approximately 1
assert np.allclose(weights_sum, 1.0, atol=1e-6), "Attention weights don't sum to 1!"
print("✓ Attention weights correctly sum to 1")

In [ ]:
# Numerical correctness test with simple example
print("Simple Correctness Test:")
print("=" * 50)

# Create simple test case: 1 batch, 3 sequence positions, 4 dimensions
Q_test = np.array([[[1, 0, 0, 0],
                     [0, 1, 0, 0],
                     [0, 0, 1, 0]]])

K_test = np.array([[[1, 0, 0, 0],
                     [0, 1, 0, 0],
                     [0, 0, 1, 0]]])

V_test = np.array([[[1, 1, 1, 1],
                     [2, 2, 2, 2],
                     [3, 3, 3, 3]]])

output_test, weights_test = scaled_dot_product_attention(Q_test, K_test, V_test)

print("Test attention weights (first batch):")
print(weights_test[0])
print()
print("Test output (first batch):")
print(output_test[0])
print()

# Verify properties
print("Verification:")
print(f"✓ All attention weights are non-negative: {(weights_test >= 0).all()}")
print(f"✓ Attention weights sum to 1: {np.allclose(weights_test.sum(axis=-1), 1.0)}")
print(f"✓ Output is weighted combination of V: {(output_test[0, 0] >= 1).all() and (output_test[0, 0] <= 3).all()}")

## Step 5: Evaluation and Visualization

**Task**: Visualize attention patterns to understand the mechanism.

**Requirements**:
- Create a heatmap of attention weights for one sample
- Label axes appropriately (Query positions vs Key positions)
- Use a color scale to show attention strength
- Interpret the attention pattern
- Compare attention patterns across different samples in the batch

In [ ]:
# Visualize attention scores for the first sample in batch
plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0], 
            annot=True, 
            fmt='.3f', 
            cmap='YlOrRd',
            xticklabels=[f'K{i}' for i in range(seq_length)],
            yticklabels=[f'Q{i}' for i in range(seq_length)],
            cbar_kws={'label': 'Attention Weight'})

plt.title('Attention Weights Heatmap (Sample 1)', fontsize=14, fontweight='bold')
plt.xlabel('Key Positions', fontsize=12)
plt.ylabel('Query Positions', fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Each row represents how much a query position attends to each key position")
print("- Brighter colors indicate stronger attention")
print("- Each row sums to 1.0 (shown in annotations)")

In [ ]:
# Compare attention patterns across batch samples
fig, axes = plt.subplots(1, batch_size, figsize=(14, 5))

for idx in range(batch_size):
    sns.heatmap(attention_weights[idx],
                annot=True,
                fmt='.2f',
                cmap='viridis',
                ax=axes[idx],
                xticklabels=[f'K{i}' for i in range(seq_length)],
                yticklabels=[f'Q{i}' for i in range(seq_length)],
                cbar_kws={'label': 'Attention'})
    axes[idx].set_title(f'Sample {idx + 1}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Key Positions')
    axes[idx].set_ylabel('Query Positions')

plt.tight_layout()
plt.show()

# Additional statistics
print("\nAttention Statistics:")
print("=" * 50)
for idx in range(batch_size):
    max_attention = attention_weights[idx].max()
    min_attention = attention_weights[idx].min()
    avg_attention = attention_weights[idx].mean()
    print(f"Sample {idx + 1}:")
    print(f"  Max attention: {max_attention:.4f}")
    print(f"  Min attention: {min_attention:.4f}")
    print(f"  Avg attention: {avg_attention:.4f}")
    print()

In [ ]:
# Analyze which positions attend most strongly
print("Attention Pattern Analysis:")
print("=" * 50)

for batch_idx in range(batch_size):
    print(f"\nSample {batch_idx + 1}:")
    for query_pos in range(seq_length):
        # Find the key position with maximum attention for each query
        max_key_pos = attention_weights[batch_idx, query_pos].argmax()
        max_attention_value = attention_weights[batch_idx, query_pos, max_key_pos]
        print(f"  Q{query_pos} attends most to K{max_key_pos} (weight: {max_attention_value:.4f})")

## Bonus: Causal Masking

**Task**: Implement and test causal (autoregressive) masking.

**Requirements**:
- Create a causal mask that prevents positions from attending to future positions
- Apply the mask in the attention function
- Visualize the masked attention pattern

In [ ]:
# Create causal mask (lower triangular matrix)
causal_mask = np.tril(np.ones((seq_length, seq_length)))

print("Causal Mask:")
print(causal_mask.astype(int))
print("\n(1 = attend, 0 = masked)")
print()

# Apply attention with causal mask
output_masked, attention_weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print(f"Masked output shape: {output_masked.shape}")
print(f"Masked attention weights shape: {attention_weights_masked.shape}")

In [ ]:
# Visualize masked attention weights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original attention (no mask)
sns.heatmap(attention_weights[0],
            annot=True,
            fmt='.3f',
            cmap='YlOrRd',
            ax=axes[0],
            xticklabels=[f'K{i}' for i in range(seq_length)],
            yticklabels=[f'Q{i}' for i in range(seq_length)],
            cbar_kws={'label': 'Attention'})
axes[0].set_title('Without Causal Mask', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key Positions')
axes[0].set_ylabel('Query Positions')

# Masked attention
sns.heatmap(attention_weights_masked[0],
            annot=True,
            fmt='.3f',
            cmap='YlOrRd',
            ax=axes[1],
            xticklabels=[f'K{i}' for i in range(seq_length)],
            yticklabels=[f'Q{i}' for i in range(seq_length)],
            cbar_kws={'label': 'Attention'})
axes[1].set_title('With Causal Mask', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Key Positions')
axes[1].set_ylabel('Query Positions')

plt.tight_layout()
plt.show()

print("\nObservation:")
print("- With causal masking, each position can only attend to itself and previous positions")
print("- The upper triangle is now zero (no future information)")
print("- This is essential for autoregressive models (e.g., GPT)")
print("- Each row still sums to 1.0, but only over non-masked positions")

In [ ]:
# Verify masking correctness
print("Causal Masking Verification:")
print("=" * 50)

# Check that upper triangle is zero
for i in range(seq_length):
    for j in range(i + 1, seq_length):
        assert attention_weights_masked[0, i, j] == 0, f"Position ({i},{j}) should be masked!"

print("✓ Upper triangle is correctly masked (all zeros)")
print("✓ Each query position only attends to current and past positions")
print(f"✓ Attention weights still sum to 1: {np.allclose(attention_weights_masked.sum(axis=-1), 1.0)}")